# 01 · Data Collection
**Project:** Sentiment analysis of Grab Malaysia (BAA5113 Web & Social Media Analytics)

This notebook gathers all the raw data for the project. It has two parts:

| Section | Question it helps answer | Source |
|---|---|---|
| **A. Web performance** | How fast, accessible and search-friendly are Grab's Malaysian websites? | Google PageSpeed Insights API + reading each homepage's HTML |
| **B. Social media** | What are Malaysians saying about Grab? | Reddit, X (Twitter) and Threads, collected through Apify |

**Before running:** make sure the `.env` file contains `PAGESPEED_API_KEY` and `APIFY_TOKEN`.
All settings (URLs, search terms, how many posts) are in `config.yaml`.

> The real code lives in `src/collect.py`. This notebook just calls it and shows the results.

> **Later collections (17–18 Nov 2026):** X round 3 with Malay keywords, Threads round 2, Facebook and TikTok comments, and Malay posts restored from the saved raw files are documented in **`01b_extra_data_collection.ipynb`** (Apify actors, search terms, dates and item counts).

In [1]:
# Setup: make the project's src/ folder importable and load settings
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
from src import collect

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 200)

cfg = collect.load_config()
log = collect.setup_logging("01_data_collection")

# Switches. False = just LOAD the saved results (safe, free).
# True  = collect fresh data. Section B costs Apify credit, so only turn it on deliberately.
RUN_SECTION_A = False
RUN_SECTION_B = False
# Which social platforms to collect: None = all enabled in config.yaml, or e.g. ["reddit"]
PLATFORMS = None
print("Pilot mode:", cfg["social"]["pilot_mode"])

Pilot mode: False


---
## Section A · Web performance

### A1. PageSpeed Insights (PSI)
Google's PSI tool loads each page on a simulated phone (**mobile**) and laptop (**desktop**) and scores it out of 100.
Scores move around from test to test, so we test each URL **3 times per device** and report the **median**.

What the metrics mean:

| Metric | Plain-English meaning | "Good" threshold |
|---|---|---|
| **Performance score** | Overall speed grade (0–100) | ≥ 90 |
| **FCP** (First Contentful Paint) | When the first text/image appears | ≤ 1.8 s |
| **LCP** (Largest Contentful Paint) | When the main content has loaded | ≤ 2.5 s |
| **TBT** (Total Blocking Time) | How long the page is frozen and can't respond | ≤ 200 ms |
| **CLS** (Cumulative Layout Shift) | How much things jump around while loading | ≤ 0.1 |
| **Speed Index** | How quickly the visible page fills in | ≤ 3.4 s |
| **INP** (field only) | How quickly the page reacts to real users' taps/clicks | ≤ 200 ms |

**Lab** metrics come from Google's simulated test. **Field** metrics (CrUX) come from real Chrome users over the past 28 days, and only exist if the site has enough traffic.
If a specific page has no field data we fall back to the whole website ("origin"); the column `field_data_source` says which was used.

Every raw API response is saved in `data/web/` so results can be re-checked later.

In [2]:
# Section A results live in the "website and app analysis" folder
web_tables = collect.project_path(cfg["paths"]["web_tables"])

if RUN_SECTION_A:
    psi_runs, psi_failures = collect.run_pagespeed_batch(cfg, log)
    psi_runs.to_csv(web_tables / "pagespeed_runs.csv", index=False)
    print(f"{len(psi_runs)} successful tests, {len(psi_failures)} failed")
    if psi_failures:
        display(pd.DataFrame(psi_failures))
else:
    psi_runs = pd.read_csv(web_tables / "pagespeed_runs.csv")

psi_runs.drop(columns=["raw_file", "lighthouse_version"]).head(10)

,url,strategy,run,test_datetime,performance_score,accessibility_score,best_practices_score,seo_score,lab_fcp_ms,lab_lcp_ms,...,field_lcp_ms,field_lcp_category,field_inp_ms,field_inp_category,field_cls,field_cls_category,field_overall_category,final_url,runtime_error,lighthouse_fetch_time
0,https://dineout.grab.com/my,desktop,1,2026-11-04T21:48:58.417Z,94,91,96,100,326.396560,1410.231369,...,1312,FAST,62,FAST,0.00,FAST,FAST,https://dineout.grab.com/my/en,NaN,2026-11-04T21:48:58.417Z
1,https://dineout.grab.com/my,desktop,2,2026-11-04T21:51:39.609Z,82,91,96,100,356.393612,1144.372319,...,1312,FAST,62,FAST,0.00,FAST,FAST,https://dineout.grab.com/my/en,NaN,2026-11-04T21:51:39.609Z
2,https://dineout.grab.com/my,desktop,3,2026-11-04T21:54:21.877Z,100,88,96,100,332.248658,332.248658,...,1312,FAST,62,FAST,0.00,FAST,FAST,https://dineout.grab.com/my/en,NaN,2026-11-04T21:54:21.877Z
3,https://dineout.grab.com/my,mobile,1,2026-11-04T21:48:41.615Z,70,95,96,100,1357.297136,11234.770542,...,2839,AVERAGE,160,FAST,0.04,FAST,AVERAGE,https://dineout.grab.com/my/en,NaN,2026-11-04T21:48:41.615Z
4,https://dineout.grab.com/my,mobile,2,2026-11-04T21:51:21.639Z,63,95,96,100,2278.691490,11288.252866,...,2839,AVERAGE,160,FAST,0.04,FAST,AVERAGE,https://dineout.grab.com/my/en,NaN,2026-11-04T21:51:21.639Z
5,https://dineout.grab.com/my,mobile,3,2026-11-04T21:55:52.212Z,70,95,96,100,1357.294272,10907.993127,...,2839,AVERAGE,160,FAST,0.04,FAST,AVERAGE,https://dineout.grab.com/my/en,NaN,2026-11-04T21:55:52.212Z
6,https://express.grab.com/ms,desktop,1,2026-11-04T21:48:23.235Z,59,81,96,92,860.688930,2350.255718,...,805,FAST,45,FAST,0.00,FAST,FAST,https://express.grab.com/ms,NaN,2026-11-04T21:48:23.235Z
7,https://express.grab.com/ms,desktop,2,2026-11-04T21:51:04.655Z,65,81,96,92,828.695082,2332.780329,...,805,FAST,45,FAST,0.00,FAST,FAST,https://express.grab.com/ms,NaN,2026-11-04T21:51:04.655Z
8,https://express.grab.com/ms,desktop,3,2026-11-04T21:53:58.626Z,53,81,96,92,928.687705,2446.149874,...,805,FAST,45,FAST,0.00,FAST,FAST,https://express.grab.com/ms,NaN,2026-11-04T21:53:58.626Z
9,https://express.grab.com/ms,mobile,1,2026-11-04T21:48:06.089Z,47,81,96,92,3456.087415,11855.349662,...,1566,FAST,112,FAST,0.00,FAST,FAST,https://express.grab.com/ms,NaN,2026-11-04T21:48:06.089Z


**Median table:** one row per URL × device. These are the numbers to quote in the report.

In [3]:
psi_median = collect.pagespeed_median_table(psi_runs)
psi_median.to_csv(web_tables / "pagespeed_median.csv", index=False)

# Compact view of the key columns
key_cols = ["url", "strategy", "n_runs", "performance_score", "accessibility_score",
            "best_practices_score", "seo_score", "lab_fcp_ms", "lab_lcp_ms", "lab_tbt_ms",
            "lab_cls", "lab_speed_index_ms", "field_data_source", "field_lcp_ms",
            "field_inp_ms", "field_cls", "field_overall_category"]
psi_median[key_cols]

,url,strategy,n_runs,performance_score,accessibility_score,best_practices_score,seo_score,lab_fcp_ms,lab_lcp_ms,lab_tbt_ms,lab_cls,lab_speed_index_ms,field_data_source,field_lcp_ms,field_inp_ms,field_cls,field_overall_category
0,https://dineout.grab.com/my,desktop,3,94.0,91.0,96.0,100.0,332.249,1144.372,112.5,0.007,878.071,page,1312.0,62.0,0.00,FAST
1,https://dineout.grab.com/my,mobile,3,70.0,95.0,96.0,100.0,1357.297,11234.771,148.5,0.001,4897.298,page,2839.0,160.0,0.04,AVERAGE
2,https://express.grab.com/ms,desktop,3,59.0,81.0,96.0,92.0,860.689,2350.256,535.0,0.000,2438.640,page,805.0,45.0,0.00,FAST
3,https://express.grab.com/ms,mobile,3,43.0,81.0,96.0,92.0,3493.075,12173.647,784.5,0.000,7314.409,page,1566.0,112.0,0.00,FAST
4,https://food.grab.com/my/,desktop,3,76.0,76.0,96.0,100.0,372.573,1705.478,369.0,0.000,834.081,page,2452.0,114.0,0.48,SLOW
5,https://food.grab.com/my/,mobile,3,15.0,82.0,96.0,100.0,1666.502,24657.518,2338.5,0.456,17675.394,page,6537.0,427.0,1.00,SLOW
6,https://mart.grab.com/my,desktop,3,74.0,80.0,96.0,100.0,330.311,1511.443,535.0,0.000,1129.999,page,1482.0,70.0,0.00,AVERAGE
7,https://mart.grab.com/my,mobile,3,54.0,75.0,92.0,85.0,1109.492,6238.143,959.0,0.000,5941.584,page,2718.0,241.0,0.07,AVERAGE
8,https://www.grab.com/my/,desktop,3,65.0,82.0,77.0,85.0,707.000,3361.791,554.0,0.051,2190.404,page,1682.0,96.0,0.04,FAST
9,https://www.grab.com/my/,mobile,3,46.0,81.0,77.0,85.0,3021.000,30340.000,648.5,0.000,12358.996,page,2253.0,371.0,0.00,AVERAGE


### A2. On-page SEO check
We download each homepage's HTML (after checking its `robots.txt` allows it) and look at the basic SEO elements:

- **Title** and **meta description**, plus their lengths. Google shows roughly 50–60 characters of a title and 150–160 of a description.
- **H1 / H2 headings**: a page should normally have exactly one H1.
- **Canonical** tag: tells Google which URL is the "official" version.
- **hreflang** tags: tell Google which language/country each version is for (e.g. `en-MY`, `ms-MY`).
- **Images without `alt` text**: hurts accessibility and image SEO.
- **Viewport** tag: needed for the page to display properly on phones.

*Note:* some Grab pages build their content with JavaScript after loading. We only see the HTML the server sends,
so a missing H1 here may mean "added later by JavaScript" rather than "missing entirely".
For example, `express.grab.com` sends an almost empty page shell and builds everything with JavaScript.
If a site blocks automated requests (HTTP 403), we record that and do **not** try to get around the block.
The PSI SEO score from A1 still covers those pages, because Google runs a real browser.

In [4]:
if RUN_SECTION_A:
    seo = collect.scrape_onpage_seo_many(cfg, log)
    seo.to_csv(web_tables / "onpage_seo.csv", index=False)
else:
    seo = pd.read_csv(web_tables / "onpage_seo.csv")

seo_view = ["url", "robots_allowed", "http_status", "title", "title_length",
            "meta_description_length", "h1_count", "h2_count", "canonical",
            "hreflang_count", "img_total", "img_missing_alt", "has_viewport"]
seo[[c for c in seo_view if c in seo.columns]]

,url,robots_allowed,http_status,title,title_length,meta_description_length,h1_count,h2_count,canonical,hreflang_count,img_total,img_missing_alt,has_viewport
0,https://www.grab.com/my/,True,200,Grab. The Everyday Everything App | Grab,40.0,129.0,1.0,0.0,https://www.grab.com/my/,18.0,7.0,4.0,True
1,https://food.grab.com/my/,True,200,Food Delivery Malaysia - Promos & Menu | GrabFood MY,52.0,150.0,0.0,2.0,https://food.grab.com/my/en/,14.0,23.0,0.0,True
2,https://mart.grab.com/my,True,403,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,https://express.grab.com/ms,True,200,NaN,0.0,0.0,0.0,0.0,NaN,6.0,0.0,0.0,True
4,https://dineout.grab.com/my,True,200,Restaurant Deals in Malaysia | Grab Dine Out,44.0,148.0,1.0,1.0,https://dineout.grab.com/my/en,12.0,16.0,0.0,True


---
## Section B · Social media (Apify)

We use three ready-made Apify "actors" (scrapers), set in `config.yaml`:

| Platform | Actor | How we search |
|---|---|---|
| Reddit | `fatihtahta/reddit-scraper-search-fast` | Inside r/malaysia, r/kualalumpur, r/Bolehland and r/penang only, with up to 30 comments per post |
| X (Twitter) | `xquik/x-tweet-scraper` | Whole of X, one search per query (works on the Apify Free plan) |
| Threads | `futurizerush/meta-threads-scraper` | Threads keyword search |

**Search terms** (editable in `config.yaml`) include English and Malay phrases, e.g. *grab mahal* ("Grab is expensive")
and *grab lambat* ("Grab is slow"). Only posts from the **last 12 months** are collected.

**Pilot mode** (`social.pilot_mode: true`) collects a small sample (10–20 posts per query)
so we can check the posts are actually about Grab before paying for the full run.

**Budget:** hard limit of **US$5 per month** (`social.monthly_budget_usd`). Before every actor run the code checks
Apify's real month-to-date usage and lowers or skips the run so the total never goes over. The Apify Free plan gives US$5 of credit per month, shared by all three platforms. Target ≈ 3,000 items:
Reddit ≈ 1,260 (US$1.90), X ≈ 1,300 (US$0.20), Threads ≈ 450 (US$1.15). Each platform has its own spending cap in `config.yaml`.

**Privacy:** usernames are replaced with an irreversible code (`author_hash`), and saved links don't contain usernames.
The raw scraper files in `data/raw/` do contain usernames, so that folder is never shared or committed.

The cell below shows the plan and the **maximum** possible cost before anything is spent.

In [5]:
PILOT = cfg["social"]["pilot_mode"]
plan = collect.build_run_plan(cfg, PILOT, PLATFORMS)
plan_cost = collect.estimate_cost(cfg, plan)
print("Mode:", "PILOT" if PILOT else "FULL", "| platforms:", PLATFORMS or "all")
for p in plan:
    print(f"  {p['platform']:8} {p['label']:15} window {collect.date_window(cfg, p['platform'])}")
display(plan_cost.groupby("platform")[["max_items", "max_cost_usd"]].sum().round(2))
print(f"Worst-case total: US${plan_cost.max_cost_usd.sum():.2f}")

Mode: FULL | platforms: all
  reddit   r/malaysia      window ('2025-11-17', '2026-11-17')
  twitter  grab (JPJ OR APAD OR SOCSO OR PERKESO OR "gig workers" OR "gig worker" OR "Gig Workers Bill" OR "Anthony Loke" OR LHDN OR SST OR "e-hailing" OR ehailing) lang:en -$GRAB -Philippines -Indonesia -Singapore [2025-11-17..2026-05-18] window ('2025-11-17', '2026-11-17')
  twitter  grab (JPJ OR APAD OR SOCSO OR PERKESO OR "gig workers" OR "gig worker" OR "Gig Workers Bill" OR "Anthony Loke" OR LHDN OR SST OR "e-hailing" OR ehailing) lang:en -$GRAB -Philippines -Indonesia -Singapore [2026-05-18..2026-11-17] window ('2025-11-17', '2026-11-17')
  twitter  (grabfood OR "grab food") ("nasi lemak" OR mamak OR "roti canai" OR "teh tarik" OR "char kuey teow" OR "nasi kandar" OR durian OR "mee goreng" OR "ayam goreng" OR satay OR rendang) lang:en -$GRAB -Singapore -Indonesia -Jakarta -Philippines [2025-11-17..2026-05-18] window ('2025-11-17', '2026-11-17')
  twitter  (grabfood OR "grab food") ("nasi l

,max_items,max_cost_usd
platform,,
reddit,9300,13.86
threads,450,1.20
twitter,1150,0.17


Worst-case total: US$15.23


### Run the collection
Each actor run is retried up to 3 times if it fails. If one platform fails completely, the others still run
and the failure is listed at the end. Raw output goes to `data/raw/`. Each platform gets its own table in `data/interim/` (e.g. `posts_reddit.csv`),
and `posts_raw_combined.csv` is rebuilt from all of them, so collecting one platform never deletes another.

In [6]:
if RUN_SECTION_B:
    result = collect.collect_social(cfg, log, pilot=PILOT, platforms=PLATFORMS)
    posts = result["posts"]
    print("Files written:", result["files"])
    if len(result["failures"]):
        print("\nFAILED RUNS:")
        display(result["failures"])
else:
    fname = "posts_raw_combined_pilot.csv" if PILOT else "posts_raw_combined.csv"
    posts = pd.read_csv(collect.PROJECT_ROOT / cfg["paths"]["interim"] / fname)
    result = None
print(f"Total posts collected: {len(posts)}")

Total posts collected: 4782


### Summary: posts per platform and query, plus cost

In [7]:
# Items per platform, split into posts vs comments/replies
display(pd.crosstab(posts["platform"], posts["is_comment"].map({False: "posts", True: "comments/replies"}),
                    margins=True, margins_name="TOTAL"))
# Items per search query (queries differ by platform, see config.yaml)
display(posts.groupby(["platform", "query"]).size().rename("items").reset_index())
if result is not None:
    cost = result["cost"]
    display(cost)
    print(f"Estimated Apify cost: US${cost.estimated_cost_usd.sum():.2f}"
          f"  (Apify-reported platform usage: US${cost.apify_reported_usd.fillna(0).sum():.2f})")

is_comment,comments/replies,posts,TOTAL
platform,,,
reddit,1372,124,1496
threads,90,287,377
twitter,919,1990,2909
TOTAL,2381,2401,4782


,platform,query,items
0,reddit,r/Bolehland: Grab Malaysia,139
1,reddit,r/Bolehland: Grab app,18
2,reddit,r/Bolehland: Grab customer service,93
3,reddit,r/Bolehland: Grab delivery,69
4,reddit,r/Bolehland: Grab driver,61
...,...,...,...
66,twitter,"grab (Malaysia OR Malaysian OR KL OR ""Kuala Lumpur"" OR Penang OR Johor OR JB OR Selangor OR KLIA) lang:en -$GRAB -st...",170
67,twitter,grab (RM OR ringgit) (fare OR fares OR price OR charged OR charge OR expensive OR cheap OR paid OR surge OR fee OR f...,34
68,twitter,"grab (expensive OR surge OR cancelled OR cancel OR late OR ""customer service"" OR refund OR scam) (Malaysia OR KL OR ...",31
69,twitter,"grab (foodpanda OR shopeefood OR ""shopee food"" OR gojek OR indrive OR maxim OR ""airasia ride"" OR lalamove OR ""e-hail...",44


### Relevance check: 15 random posts per platform
Read these and ask: *is this post really about Grab in Malaysia?* If many aren't (e.g. "grab a coffee"),
we should change the search terms in `config.yaml` before the full run.

In [8]:
for platform, grp in posts.groupby("platform"):
    sample = grp.sample(min(15, len(grp)), random_state=42)
    print(f"\n===== {platform.upper()} ({len(grp)} posts) =====")
    display(sample[["query", "created_at", "likes", "is_comment", "text"]]
            .assign(text=lambda d: d.text.str.replace(r"\s+", " ", regex=True).str.slice(0, 280)))


===== REDDIT (1496 posts) =====


,query,created_at,likes,is_comment,text
918,r/penang: Grab driver,2026-05-09T23:41:36Z,1.0,True,"Butterworth to Ipoh train ticket can only book 48 hours in advance, see: https://www.reddit.com/r/penang/comments/1s..."
1350,r/malaysia: GrabPay,2026-05-04T05:03:41Z,33.0,True,"Yes, I've seen this type of reporting a lot in Malaysian businesses. Typically the outlet/store manager will send it..."
1061,r/kualalumpur: Grab Malaysia,2026-05-06T08:08:29Z,11.0,True,Use grab app . It wiill save you both time and money
532,r/Bolehland: Grab customer service,2026-06-11T22:07:16Z,1.0,True,KFC bolehl!
513,r/Bolehland: Grab customer service,2026-08-15T00:23:38Z,-1.0,True,NaN
1090,r/kualalumpur: Grab Malaysia,2026-01-04T11:17:55Z,2.0,True,Also KL Sentral has a luggage locker. It's a bit hidden but it's there.
482,r/kualalumpur: Grab driver,2025-12-10T10:26:02Z,10.0,True,"That rating says it all. You can go and accept for as low as 4.9 stars if you're in a hurry but below that, those dr..."
693,r/Bolehland: Grab Malaysia,2025-12-16T13:37:35Z,1.0,True,Lmao damn. Dude can't understand the concept of an example. Immediate seethe and mald at the word chinese. Stereotyp...
374,r/kualalumpur: Grab driver,2026-06-18T21:54:17Z,2.0,True,Not creepy and suspicious at all... /s
244,r/malaysia: Grab driver,2025-11-22T05:23:49Z,5.0,True,These ones probably kerja more than 10 hours a day but of course it’s very very tiring. They are also on the high ti...



===== THREADS (377 posts) =====


,query,created_at,likes,is_comment,text
4691,grab penang,2026-11-03T14:40:20Z,1.0,False,*WANT TO SELL* IPHONE 16pro 128GB - Battery health 89% 🔋 - Condition 9/10 ✅ - Function semua okay 👍 - COD area Kedah...
4663,grab KL,2026-11-04T00:54:39Z,3.0,False,Hi everyone! I am looking for a room to rent Preferences: -Preferably a single room (non-sharing). -Near PPUM or wit...
4667,Grab Malaysia,2026-11-04T00:21:15Z,63.0,False,"Malaysia’s healthcare system is structurally broken, and vulnerable patients are paying the price with their zero hu..."
4550,grab cancel,2026-11-04T10:03:38Z,875.0,False,Democrats appear to have more hate with victory than they had with defeat.
4460,Grab Malaysia,2026-11-05T03:44:05Z,6.0,False,"Looking to collaborate with fellow Dietitians, Nutritionists, Doctors, Pharmacists and healthcare professionals in M..."
4498,grab penang,2026-11-05T00:25:58Z,6.0,False,🚨 URGENT: EVENT CREW REPLACEMENT NEEDED – 1 VACANCY! 🚨 We urgently need 1 replacement crew member for the AmAssuranc...
4744,grab KL,2026-11-03T05:00:30Z,16.0,False,"TOYS R US KL EAST MALL 30th TCG drop at 3/11/26, 12.45pm"
4487,Grab Malaysia; grabfood malaysia,2026-11-05T02:03:00Z,157.0,False,Bruh Malaysia crowd really complaining on standing at seating areas? It's a concert. Concerts I went abroad had peop...
4769,GrabMY,2026-11-02T08:19:17Z,3006.0,True,"Salam puan, kami minta maaf atas kekeliruan maklumat sebelum ini. Arahan penghantaran (door to door) masih berkuat k..."
4553,grabfood malaysia,2026-11-04T09:33:49Z,146.0,False,Calling all Malaysian Lifestyle & Beauty KOLs! I’m currently on the lookout for a KOL who would be interested in joi...



===== TWITTER (2909 posts) =====


,query,created_at,likes,is_comment,text
2363,"(grabfood OR ""grab food"") (Malaysia OR KL OR RM OR Penang OR Johor OR rider OR delivery OR order) lang:en -$GRAB -Ph...",2026-09-09T09:45:11Z,81.0,False,"After living in China, I have a running list of what I miss. Here's two more of them; 📱 Payment services What makes ..."
2717,"(grabfood OR ""grab food"") (Malaysia OR KL OR RM OR Penang OR Johor OR rider OR delivery OR order) lang:en -$GRAB -Ph...",2026-02-10T11:51:17Z,192.0,False,Fandez ngepoin Rose order apa di Grab food 🤭 He needs comfort food (and hugs) for sure! https://t.co/GiAgiEhWL7
2370,"(grabfood OR ""grab food"") (Malaysia OR KL OR RM OR Penang OR Johor OR rider OR delivery OR order) lang:en -$GRAB -Ph...",2026-09-07T01:30:34Z,7.0,False,"If you grew up seeing Jollibee almost everywhere, you may remember a strange little version of the fast-food giant: ..."
1850,grab (mahal OR lambat OR surge OR cancel) (tak OR nak OR dah OR lah OR RM) -gak -nggak -banget -udah -aja -gw -gue -...,2026-04-19T02:21:46Z,6.0,False,"Laluan train kita semakin convenient, tapi nak ke station tu bus lambat, grab mahal, takde laluan pejalan kaki, park..."
2095,"grab (Malaysia OR Malaysian OR KL OR ""Kuala Lumpur"" OR Penang OR Johor OR JB OR Selangor OR KLIA) lang:en -$GRAB -st...",2026-10-31T03:12:19Z,21.0,False,Grab your tickets now and 𝗗𝗢 𝗜𝗧 𝗙𝗢𝗥 𝗦𝗘𝗟𝗔𝗡𝗚𝗢𝗥 🫡 See you tomorrow #RedGiants! 💊 🎟️ https://t.co/9xgjw7E0CD 🔗 #Selangor...
4055,"(grab OR grabcar OR grabfood) (Ipoh OR Klang OR ""Shah Alam"" OR Subang OR PJ OR ""Petaling Jaya"" OR Cyberjaya OR Putra...",2026-09-02T12:33:47Z,2.0,True,@___pkpolZ ก้อยกไปเลยสิ FREENBECKY WITH GRABFOOD #GrabFoodถูกกว่าชัวร์xฟรีนเบคกี้
1805,grab (mahal OR lambat OR surge OR cancel) (tak OR nak OR dah OR lah OR RM) -gak -nggak -banget -udah -aja -gw -gue -...,2026-09-26T11:28:01Z,3.0,False,tak expect sampai stesen maghrib pastu grab mahal pulak 😭😭
3651,"grab (foodpanda OR shopeefood OR ""shopee food"" OR gojek OR indrive OR maxim OR ""airasia ride"" OR lalamove OR ""e-hail...",2026-05-05T12:43:26Z,4.0,True,"wts lfb bts ph ✨Tinytan x Mcdo toy figurines Mint condition Only RM no card, no box for all Php 75 each Comment or d..."
3833,"(grab OR grabcar OR grabfood) (Ipoh OR Klang OR ""Shah Alam"" OR Subang OR PJ OR ""Petaling Jaya"" OR Cyberjaya OR Putra...",2026-01-14T16:56:19Z,76.0,False,CATLUMPURR IS CLOSER THAN YOU THINK Fail to plan Plan to fail Here’s the practical lens most people aren’t talking a...
1975,"""Grab Malaysia"" -$GRAB",2026-06-12T10:17:05Z,43.0,True,@NewsBFM ahhhhh so basically zus and wanda buat promo main2 grab tetiba masuk 🙃🙃 https://t.co/KXng64pClC https://t.c...


### Posts added by hand
After cleaning, the dataset was 7 items short of our 2,000 minimum, and the Apify budget was used up.
The team found 7 more relevant Threads posts by hand and shared their links. Their content was copied into
`data/raw/manual_threads_posts_20261106.json`, which is git-ignored because it holds usernames.
`collect.add_manual_posts()` normalised them the same way as the scraped posts (hashed author, username-free link)
and flagged them `manually_added = True`. Running it again won't create duplicates.

In [9]:
# Free: no Apify run. Re-adding is safe because duplicates are skipped.
collect.add_manual_posts(cfg, "data/raw/manual_threads_posts_20261106.json")
posts = pd.read_csv(collect.PROJECT_ROOT / cfg["paths"]["interim"] / "posts_raw_combined.csv")
manual = posts[posts["manually_added"].fillna(False).astype(bool)]
print(f"{len(manual)} hand-added posts; combined total now {len(posts):,}")
manual[["platform", "created_at", "likes", "replies", "text"]]

7 hand-added posts; combined total now 4,782


,platform,created_at,likes,replies,text
4775,threads,2026-11-03T00:00:00Z,5.0,7.0,why can't I use the 20% off vouchers for grab signatures today?
4776,threads,2026-09-04T00:00:00Z,5100.0,300.0,"Hello @grabmy , what is this ? is this how grab drivers are allowed to act?"
4777,threads,2026-04-05T00:00:00Z,1000.0,192.0,HELLO @grabmy I booked a Grab for 7:30 AM. The driver arrived earlier than the scheduled time and cancelled the book...
4778,threads,2026-01-25T00:00:00Z,567.0,19.0,"I have seen so many Threads sharing stories about kind Grab drivers, and today I finally experienced one myself 🥰🤍"
4779,threads,2026-07-16T00:00:00Z,160.0,31.0,WHY SOME GRAB DRIVERS ARE LIKE THIS.Steal money easily like this.There is no toll at all and look what he has done.🙄...
4780,threads,2026-07-17T00:00:00Z,79.0,13.0,woww @grabmy can't believe your grab rider asked me to cancel my order just because he wants to go home? I'm so amaz...
4781,threads,2026-10-09T00:00:00Z,9.0,15.0,"Hey @grabmy , I would like to clarify your toll fee policy for trips originating from Genting Highlands. My driver r..."


---
**Next step:** `02_data_cleaning.ipynb` reads `data/interim/posts_raw_combined.csv` (from the full run).